# Task 1

## Setup and Imports

In [12]:
%pip install trl==0.8.6

  Using cached docstring_parser-0.17.0-py3-none-any.whl.metadata (3.5 kB)
Using cached docstring_parser-0.17.0-py3-none-any.whl (36 kB)
  Attempting uninstall: trl
    Found existing installation: trl 1.0.0
    Uninstalling trl-1.0.0:
      Successfully uninstalled trl-1.0.0
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import torch
import random
import re
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from datasets import Dataset
from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead

device = 'cuda' if torch.cuda.is_available() else 'cpu'

c:\Users\Joseph\Desktop\SchoolCode\auburn\Gen-AI-Assignments\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Stage 1: Create a synthetic reasoning dataset

In [ ]:
def generate_dataset():
    random.seed(42)
    data = []
    
    # c. Operand ranges
    operands = list(range(2, 21))
    
    # a. 1,200 examples
    while len(data) < 1200:
        a = random.choice(operands)
        b = random.choice(operands)
        c = random.choice(operands)
        t = random.randint(1, 4)
        
        # b. Task format templates
        # f. Reasoning trace format
        if t == 1:
            expr = f"({a} + {b}) - {c}"
            s1 = a + b
            r_str = f"{a} + {b} = {s1}. {s1} - {c} = {s1 - c}."
            ans = s1 - c
        elif t == 2:
            if a < b:
                a, b = b, a
            expr = f"({a} - {b}) + {c}"
            s1 = a - b
            r_str = f"{a} - {b} = {s1}. {s1} + {c} = {s1 + c}."
            ans = s1 + c
        elif t == 3:
            expr = f"{a} + ({b} * {c})"
            s1 = b * c
            r_str = f"{b} * {c} = {s1}. {a} + {s1} = {a + s1}."
            ans = a + s1
        elif t == 4:
            expr = f"({a} * {b}) - {c}"
            s1 = a * b
            r_str = f"{a} * {b} = {s1}. {s1} - {c} = {s1 - c}."
            ans = s1 - c

        # e. Instruction format
        instruction = (
            "Instruction: Solve the arithmetic expression step by step. Use exactly two lines:\n"
            "Reasoning: ...\n"
            "Answer: <integer>\n"
            f"Expression: {expr}\n"
            "Response:\n"
        )
        response = f"Reasoning: {r_str}\nAnswer: {ans}"
        
        data.append({
            "instruction": instruction,
            "response": response,
            "text": instruction + response,
            "answer": str(ans)
        })
        
    return data

full_data = generate_dataset()

#d. Dataset splits
sft_train_data = full_data[:800]
sft_val_data = full_data[800:1000]
ppo_train_data = full_data[1000:1100]
test_data = full_data[1100:1200]

## Stage 2: Supervised Fine-Tuning (SFT)
### a. Pretrained distilgpt2 and tokenizer

In [6]:
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token
sft_model = AutoModelForCausalLM.from_pretrained("distilgpt2").to(device)

c:\Users\Joseph\Desktop\SchoolCode\auburn\Gen-AI-Assignments\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Joseph\.cache\huggingface\hub\models--distilgpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 76/76 [00:00<00:00, 16874.91it/s]


### c. Response-only loss masking

In [ ]:
def tokenize_sft(example):
    inst_tokens = tokenizer(example["instruction"], add_special_tokens=False)["input_ids"]
    resp_tokens = tokenizer(example["response"] + tokenizer.eos_token, add_special_tokens=False)["input_ids"]
    
    input_ids = inst_tokens + resp_tokens
    # Masking
    labels = [-100] * len(inst_tokens) + resp_tokens 
    
    return {"input_ids": input_ids, "labels": labels}

def data_collator(features):
    input_ids = [torch.tensor(f["input_ids"]) for f in features]
    labels = [torch.tensor(f["labels"]) for f in features]
    
    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)
    
    return {
        "input_ids": input_ids, 
        "labels": labels, 
        "attention_mask": (input_ids != tokenizer.pad_token_id).long()
    }

### b. SFT training split

In [8]:
sft_train_dataset = Dataset.from_list(sft_train_data).map(tokenize_sft)
sft_val_dataset = Dataset.from_list(sft_val_data).map(tokenize_sft)

training_args = TrainingArguments(
    output_dir="./sft_results",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    logging_steps=10
)

trainer = Trainer(
    model=sft_model,
    args=training_args,
    train_dataset=sft_train_dataset,
    eval_dataset=sft_val_dataset,
    data_collator=data_collator
)

trainer.train()
trainer.save_model("./sft_model_saved")

Map: 100%|██████████| 200/200 [00:00<00:00, 6152.94 examples/s]
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,0.485255,0.440996


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.00it/s]


## Stage 3: PPO-based alignment
### a. PPO starting from SFT checkpoint
### c. Frozen reference copy

In [9]:
ppo_model = AutoModelForCausalLMWithValueHead.from_pretrained("./sft_model_saved").to(device)
ref_model = AutoModelForCausalLMWithValueHead.from_pretrained("./sft_model_saved").to(device)

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 11692.73it/s]


### b. PPO training examples

In [10]:
ppo_config = PPOConfig(
    batch_size=8,
    mini_batch_size=8,
    learning_rate=1.41e-5,
    ppo_epochs=1,
)

ppo_dataset = Dataset.from_list(ppo_train_data).map(lambda x: {
    "query": tokenizer.encode(x["instruction"], return_tensors="pt")[0],
    "answer": x["answer"]
})

def collator_ppo(data):
    return {
        "query": [d["query"] for d in data],
        "answer": [d["answer"] for d in data]
    }

ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=ppo_model,
    ref_model=ref_model,
    tokenizer=tokenizer,
    dataset=ppo_dataset,
    data_collator=collator_ppo
)

Map: 100%|██████████| 100/100 [00:00<00:00, 5881.95 examples/s]


### d. Reward function
### e. Parse output

In [11]:
def compute_reward(response_text, gold_answer):
    # e. Parse output using the line Answer: <integer>
    match = re.search(r"Answer:\s*(-?\d+)", response_text)
    if match:
        parsed_ans = match.group(1)
        # d. Reward function calculation
        if parsed_ans == gold_answer:
            return 1.0
        return 0.2
    return 0.0

### f. Run at least 100 PPO optimization steps

In [ ]:
generation_kwargs = {
    "min_length": -1,
    "top_k": 0.0,
    "top_p": 1.0,
    "do_sample": True,
    "pad_token_id": tokenizer.pad_token_id,
    "max_new_tokens": 40
}

steps_per_epoch = len(ppo_dataset) // ppo_config.batch_size
epochs_to_run = max(1, 100 // max(1, steps_per_epoch))

for epoch in range(epochs_to_run):
    for i in range(steps_per_epoch):
        batch_idx = i * ppo_config.batch_size
        batch = [ppo_dataset[j] for j in range(batch_idx, batch_idx + ppo_config.batch_size)]
        
        queries = [torch.tensor(b["query"], device=device) for b in batch]
        gold_answers = [b["answer"] for b in batch]
        
        responses = ppo_trainer.generate(queries, **generation_kwargs)
        
        rewards = []
        for q, resp, gold in zip(queries, responses, gold_answers):
            resp_only = resp[len(q):]
            decoded_resp = tokenizer.decode(resp_only, skip_special_tokens=True)
            r = compute_reward(decoded_resp, gold)
            rewards.append(torch.tensor(r, dtype=torch.float, device=device))
            
        stats = ppo_trainer.step(queries, responses, rewards)

ppo_model.save_pretrained("./ppo_model_saved")

c:\Users\Joseph\Desktop\SchoolCode\auburn\Gen-AI-Assignments\.venv\Lib\site-packages\trl\trainer\ppo_trainer.py:1289: UserWarning: KL divergence is starting to become negative: -34.26 - this might be a precursor for failed training. sometimes this happens because the generation kwargs are not correctly set. Please make sure that the generation kwargs are set correctly, or review your training hyperparameters.
  warnings.warn(
c:\Users\Joseph\Desktop\SchoolCode\auburn\Gen-AI-Assignments\.venv\Lib\site-packages\trl\trainer\ppo_trainer.py:1289: UserWarning: KL divergence is starting to become negative: -54.05 - this might be a precursor for failed training. sometimes this happens because the generation kwargs are not correctly set. Please make sure that the generation kwargs are set correctly, or review your training hyperparameters.
  warnings.warn(
c:\Users\Joseph\Desktop\SchoolCode\auburn\Gen-AI-Assignments\.venv\Lib\site-packages\trl\trainer\ppo_trainer.py:1289: UserWarning: KL diverg

## Stage 4: Evaluate reasoning performance
### a. Evaluate models (Base, SFT, PPO)
### b. 100 held-out test examples
### c. Report metrics
### d. Greedy decoding
### e. Show outputs for 5 test prompts

In [14]:
base_eval_model = AutoModelForCausalLM.from_pretrained("distilgpt2").to(device)
sft_eval_model = AutoModelForCausalLM.from_pretrained("./sft_model_saved").to(device)
ppo_eval_model = AutoModelForCausalLM.from_pretrained("./ppo_model_saved").to(device)

def evaluate_model(eval_model, name):
    eval_model.eval()
    exact_match = 0
    format_adherence = 0
    total_reward = 0.0
    sample_outputs = []
    
    for i, item in enumerate(test_data):
        input_ids = tokenizer.encode(item["instruction"], return_tensors="pt").to(device)
        
        with torch.no_grad():
            # d. Greedy decoding 
            outputs = eval_model.generate(
                input_ids, 
                max_new_tokens=40, 
                pad_token_id=tokenizer.pad_token_id,
                do_sample=False 
            )
        
        full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
        resp_text = full_output[len(item["instruction"]):].strip()
        
        reward = compute_reward(resp_text, item["answer"])
        total_reward += reward
        
        if reward == 1.0:
            exact_match += 1
        
        if re.search(r"Answer:\s*-?\d+", resp_text):
            format_adherence += 1
            
        if i < 5:
            sample_outputs.append((item["instruction"], resp_text))
            
    n = len(test_data)
    # c. Metrics to report
    print(f"Results for {name}:")
    print(f"Exact Match Accuracy: {exact_match/n:.2f}")
    print(f"Format Adherence Rate: {format_adherence/n:.2f}")
    print(f"Average Reward: {total_reward/n:.2f}")
    
    # e. Show outputs for 5 test prompts
    print("\nSample Outputs (First 5):")
    for inst, resp in sample_outputs:
        print("-------------")
        print(inst.strip())
        print(f"Model Output:\n{resp}")
    print("=========================================\n")

evaluate_model(base_eval_model, "Base Model")
evaluate_model(sft_eval_model, "SFT Model")
evaluate_model(ppo_eval_model, "PPO Model")

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 15194.58it/s]
GPT2LMHeadModel LOAD REPORT from: ./ppo_model_saved
Key                   | Status     |  | 
----------------------+------------+--+-
v_head.summary.weight | UNEXPECTED |  | 
v_head.summary.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Results for Base Model:
Exact Match Accuracy: 0.00
Format Adherence Rate: 0.00
Average Reward: 0.00

Sample Outputs (First 5):
-------------
Instruction: Solve the arithmetic expression step by step. Use exactly two lines:
Reasoning: ...
Answer: <integer>
Expression: 17 + (9 * 4)
Response:
Model Output:
Answer: <integer>
Expression: 17 + (9 * 4)
Response: <integer>
Expression: 17 + (9 * 4)
Response: <integer>
-------------
Instruction: Solve the arithmetic expression step by step. Use exactly two lines:
Reasoning: ...
Answer: <integer>
Expression: (13 - 9) + 3
Response:
Model Output:
Answer: <integer>
Expression: (13 - 9) + 3
Response: <integer>
Expression: (13 - 9) + 3
Response: <integer>
-------------
Instruction: Solve the arithmetic expression step by step. Use exactly two lines:
Reasoning: ...
Answer: <integer>
Expression: (18 * 13) - 14
Response:
Model Output:
Answer: <integer>
Expression: (18 * 13) - 14
Response: <integer>
Expression: (18 * 13) - 14
Response: <integer>
---------

## Stage 5: External augmentation
### a. Simple calculator-augmented system
### b. Create 20 arithmetic word problems
### c. Word problem example formatting

In [ ]:
word_problems = []
for i in range(20):
    a = random.randint(2, 20)
    b = random.randint(2, 20)
    c = random.randint(2, 20)
    t = random.randint(1, 4)
    
    #Ai helper to generate word problems
    if t == 1:
        text = f"John had {a} apples. He bought {b} more and then gave away {c}. How many apples does he have now?"
        ans = (a + b) - c
    elif t == 2:
        if a < b:
            a, b = b, a
        text = f"Sarah collected {a} coins. She lost {b} coins, and later found {c} more. How many coins does she have?"
        ans = (a - b) + c
    elif t == 3:
        text = f"A library has {a} books. They receive {b} boxes, each containing {c} books. How many books are there in total?"
        ans = a + (b * c)
    elif t == 4:
        text = f"There are {a} crates, each holding {b} melons. If {c} melons are sold, how many melons are left?"
        ans = (a * b) - c
        
    word_problems.append({"text": text, "answer": ans})

### d. Calculator tool
### e. One-line JSON tool call
### f. Python functions to execute tool

In [16]:
def calculator_tool(expression):
    try:
        return eval(expression)
    except:
        return None

def solve_with_tool(problem, model_eval):
    # e. JSON tool call prompt structure
    prompt = (
        f"Problem: {problem['text']}\n"
        "Generate a JSON tool call to solve this. "
        "Example: {\"tool\": \"calculator\", \"expression\": \"(7+8)-3\"}\n"
        "Call:"
    )
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model_eval.generate(
            input_ids, 
            max_new_tokens=30, 
            pad_token_id=tokenizer.pad_token_id,
            do_sample=False
        )
    
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    resp_text = full_output[len(prompt):].strip()
    
    try:
        match = re.search(r'\{.*?\}', resp_text)
        if match:
            tool_call = json.loads(match.group(0))
            if tool_call.get("tool") == "calculator":
                # f. Python execution
                return calculator_tool(tool_call.get("expression", ""))
    except:
        pass
    return None

def direct_answer(problem, model_eval):
    prompt = (
        "Instruction: Solve the arithmetic word problem step by step. Use exactly two lines:\n"
        "Reasoning: ...\n"
        "Answer: <integer>\n"
        f"Problem: {problem['text']}\n"
        "Response:\n"
    )
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model_eval.generate(
            input_ids, 
            max_new_tokens=40, 
            pad_token_id=tokenizer.pad_token_id,
            do_sample=False
        )
        
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    resp_text = full_output[len(prompt):].strip()
    
    match = re.search(r"Answer:\s*(-?\d+)", resp_text)
    if match:
        return int(match.group(1))
    return None

### g. Compare models
### h. Report exact-answer accuracy

In [17]:
direct_correct = 0
tool_correct = 0

for prob in word_problems:
    # 1. PPO-aligned model answering directly
    dir_ans = direct_answer(prob, ppo_eval_model)
    if dir_ans == prob["answer"]:
        direct_correct += 1
        
    # 2. Tool-augmented system
    tol_ans = solve_with_tool(prob, ppo_eval_model)
    if tol_ans == prob["answer"]:
        tool_correct += 1

n = len(word_problems)
print(f"Direct Response Accuracy: {direct_correct / n:.2f}")
print(f"Tool Augmented Accuracy: {tool_correct / n:.2f}")

Direct Response Accuracy: 0.00
Tool Augmented Accuracy: 0.00


## Stage 6: Insight Blocks

### Block 1: SFT vs PPO
Did PPO improve exact answer accuracy relative to SFT? On which decoding method was the gain larger?

No, it got worse. Evaluate metrics show SFT got 0.02 and PPO got 0.00 accuracy.

### Block 2: Reward shaping
Why is the intermediate reward of 0.2 useful? What likely happens if you use only 0/1 rewards?

It gives partial credit for correct formatting so the model can learn step-by-step. If you just use 0/1 rewards, the model almost never randomly guesses both the format and the math correctly so it learns nothing.

### Block 3: Format adherence
Did PPO increase or decrease format adherence? Why might this happen?

It decreased from 1.00 to 0.00. The small model became unstable during reinforcement learning and suffered from "policy collapse," completely forgetting the formatting it learned in SFT. We know this happened because of the ''negative KL divergence'' warning during training, which indicates the model broke away from the reference model and the training failed.

### Block 4: Failure case
Choose one test example and compare the outputs from the base model, SFT model, and PPO model. What changed?

For 17 + (9 * 4): The base model repeated the prompt. The SFT model got the structure perfect but did the math poorly. The PPO model completely broke and only spat out "Reasoning --" and stopped.

### Block 5: External augmentation
What changed quantitatively after adding the tool?

Nothing changed. Both direct accuracy and tool accuracy were 0.00 because the PPO model was broken and couldn't output the proper tool call format.